# Image classification with PyTorch

We will classify images of 12 plant species using a pretrained neural network. To keep this a short classroom exercise, we use a small archive with 10 images per species rather than the much larger original collection.

These images are a teaching subset of the [Plant Seedlings Dataset](https://vision.eng.au.dk/plant-seedlings-dataset/) by Giselsson and colleagues ([dataset paper](https://arxiv.org/abs/1711.05458)). The publisher requires citation and distributes the images under CC BY-SA; the archive includes attribution and license details. This tiny sample is for learning the mechanics, not for reporting a reliable classification benchmark.

We will fetch the image ZIP first. The next cell uses `curl` so that you can see download progress, with a connection timeout and retries. It then checks the file's SHA-256 hash and extracts `plant_seedlings_120/`. If the archive or extracted folder already exists, it skips the unnecessary work. Colab storage is temporary, so a new runtime may have to download the archive again. Later, PyTorch will **separately download the pretrained ResNet18 weights** on first use.


In [ ]:
from pathlib import Path
from hashlib import sha256
from zipfile import ZipFile
import subprocess

archive_url = "https://github.com/mojones/earth-science-ml-pytorch-colab/releases/download/v1.0/plant-seedlings-120.zip"
archive_hash = "ac87db8c3f5447208702e3ec11aa9cfc3b199eae03d38483c7a69150b6967ec1"
archive_path = Path("assets/plant-seedlings-120.zip")
if not archive_path.exists():
    archive_path = Path("plant-seedlings-120.zip")
data_dir = Path("plant_seedlings_120")

if not data_dir.exists():
    if not archive_path.exists():
        print("Downloading plant images (44 MB):", flush=True)
        partial_path = Path("plant-seedlings-120.zip.part")
        command = ["curl", "--fail", "--location", "--progress-bar",
                   "--connect-timeout", "15", "--max-time", "300",
                   "--retry", "2", "--output", str(partial_path), archive_url]
        try:
            with subprocess.Popen(command, stderr=subprocess.PIPE, text=True) as download:
                for line in download.stderr:
                    print(line.rstrip(), end="\r", flush=True)
                if download.wait() != 0:
                    raise RuntimeError("Image download failed; check the curl message above.")
            partial_path.replace(archive_path)
        except Exception:
            partial_path.unlink(missing_ok=True)
            raise
        print("\nDownload complete.", flush=True)
    else:
        print("Using existing archive:", archive_path, flush=True)

    print("Checking archive SHA-256...", flush=True)
    actual_hash = sha256(archive_path.read_bytes()).hexdigest()
    if actual_hash != archive_hash:
        raise ValueError("Image archive hash mismatch. Delete the ZIP and try again.")
    print("Extracting 120 images...", flush=True)
    with ZipFile(archive_path) as archive:
        archive.extractall(".")
    print("Extraction complete.", flush=True)
else:
    print("Using already extracted images:", data_dir, flush=True)

species_folders = sorted(p for p in data_dir.iterdir() if p.is_dir())
image_count = sum(len(list(folder.glob("*.png"))) for folder in species_folders)
assert len(species_folders) == 12 and image_count == 120
print(image_count, "images in", len(species_folders), "species folders")
print([folder.name for folder in species_folders])


In [ ]:
import torch, torchvision, numpy as np
from torch import nn
from torch.optim import Adam

print("torch", torch.__version__, "torchvision", torchvision.__version__)
print("CUDA available?", torch.cuda.is_available())


In [ ]:
print(sorted(p.name for p in (data_dir / "Shepherds Purse").glob("*.png")))


In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

# brittle Python code, do not trust for real world

# Loop through each species folder and display up to 3 images
for species in os.listdir(data_dir) :
    folder = os.path.join(data_dir, species)
    img_files = os.listdir(folder)[:3]

    images = []
    print(img_files)
    for fname in img_files:
        fpath = os.path.join(folder, fname)
        img = Image.open(fpath)
        images.append(img)

    # boring matplotlib code to show images
    fig, axes = plt.subplots(1, len(images), figsize=(8, 3))
    fig.suptitle(species, fontsize=14)
    for ax, img in zip(axes, images):
        ax.imshow(img)
        ax.axis("off")

    plt.show()


We will pick a pretrained image classifier, ResNet18, and fine-tune it for these plant images. The pre-trained weights are downloaded automatically the first time we use them (an additional download separate from our small dataset).

To use a model we need to know:
- the architecture: number of layers, types of layers, activation functions, etc;
- the weights (unless we want to train from scratch);
- the preprocessing that should be applied to the input images.

The ResNet18 [documentation](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html) has more details. We introduce a device switch here to run image training on CUDA when a GPU is available.


In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

# Colab default: select Runtime > Change runtime type > GPU first.
# For a CPU run, comment out the next line and uncomment the one below it.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print("Image model device:", device)

weights = ResNet18_Weights.DEFAULT
weights


Notice the name of the weights includes `IMAGENET1K`.

Just as with all the ML techniques we have seen, the data has to match what the model expects. If we have try to do inference with the features in different order / scaling / encoding than we have trained, it will misbehave. The same is true with image data, so the model stores the set of transformations that we have to do to the image to get it into the right format. We could probably track this information down if we had to, but it saves a lot of time to get it straight from the model itself:

In [ ]:
# resnet wants the images to be cropped to 224 x 224 pixels and have normalised RGB values
transform = weights.transforms()  
transform

Now we can do some set up to get the data arranged. In PyTorch, we use a dataset to access labelled images, then a dataloader to put them into batches. 

In the image-specific package `torchvision` there's an `ImageFolder` dataset type that assumes the images are in labelled folders, so we can just use that:

In [ ]:
# now that we know how the images should be preprocessed, we can point at the images folder:
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import random


dataset = datasets.ImageFolder(data_dir, transform=transform)
dataset

The archive already has 10 images per species. We will collect their indices explicitly, using the PyTorch `Subset` helper to work with just these images:

In [ ]:
# Choose the same 10 images per class each time, whatever file order we see.
MAX_PER_CLASS = 10
labels_list = [label for _, label in dataset.samples]
selected_indices = []
rng = random.Random(42)
for class_idx in range(len(dataset.classes)):
    cls_idxs = [i for i, label in enumerate(labels_list) if label == class_idx]
    selected_indices.extend(rng.sample(cls_idxs, MAX_PER_CLASS))

subset = Subset(dataset, selected_indices)
len(subset)


Once we start the training, all of our outputs will be numerical lables, so it will be important to keep this list of corresponding species names somewhere

In [ ]:
dataset.classes

We will keep eight images of each species for training and two for validation. This gives every class a place in both sets, which a plain random split cannot guarantee with such a small subset. Images of the same original plant at different growth stages could still appear on both sides; the validation accuracy is only a classroom illustration, not a claim of generalisation to new plants.


In [ ]:
train_indices, val_indices = [], []
for class_idx in range(len(dataset.classes)):
    these = [i for i in selected_indices if labels_list[i] == class_idx]
    train_indices.extend(these[:8])
    val_indices.extend(these[8:])

train_ds = Subset(dataset, train_indices)
val_ds = Subset(dataset, val_indices)
len(train_ds), len(val_ds)


The input transformation (crop, resizing and normalisation) otherwise runs again every epoch. There are only 120 images here, so we can apply it **once**, store the resulting tensors in memory, and make loaders from those. This is convenient for this little fixed dataset; for a large dataset or changing random augmentations, we would normally keep transformations in the loader instead.

The cached images stay in normal computer memory. The loader's `pin_memory` option, when a CUDA GPU is selected, can make transfers to the GPU more efficient. There are no worker processes to configure, which also keeps this cell portable to Windows and macOS.


In [ ]:
from torch.utils.data import TensorDataset

train_examples = [train_ds[i] for i in range(len(train_ds))]
val_examples = [val_ds[i] for i in range(len(val_ds))]

train_ds = TensorDataset(
    torch.stack([image for image, label in train_examples]),
    torch.tensor([label for image, label in train_examples]),
)
val_ds = TensorDataset(
    torch.stack([image for image, label in val_examples]),
    torch.tensor([label for image, label in val_examples]),
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,
                          pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_ds, batch_size=32,
                        pin_memory=(device.type == "cuda"))
train_loader, val_loader


Now that the data are ready to go, we can take a look at the model. We will load it straight with the pretrained weights. Just as with our model we can look at the representation:

In [ ]:
# let's take a look at the model
# the architecture is obviously way more complicated than ours
# lots of convolutional layers
# note the name/sizes of the last layer
model = resnet18(weights=weights)
model

Becuase this model has been set up with a bit more care than ours, it has names for all the layers, so we can address them by name. In particular we can refer to the final layer, called `fc` for fully connected:

In [ ]:
# we can also address this last layer by name
model.fc

This is the one that we want to fine tune. We have to do two things. 

Firstly, notice the size - since this model was originally trained on a dataset with 1000 classes, that's the number of output neurons it has. We need to change it to match the number of outputs in our dataset, which we can do just by overwriting it:

In [ ]:
model.fc = nn.Linear(
    model.fc.in_features,   # keep the same number of inputs....
    len(dataset.classes) # ...but set the number of outputs to match our dataset
)
model = model.to(device)
model

Only the final layer needs new trainable weights. The rest of the pretrained network will act as a fixed feature extractor. The model was moved to `device` above; we will move each batch of images and labels to the **same** device below. With a CUDA GPU selected, `pin_memory` and `non_blocking=True` can help transfer the cached CPU batches.

For a frozen pretrained backbone we keep the network in evaluation mode even when training the final layer: batch-normalisation running statistics should not change. This does **not** prevent gradients through the new final layer.


In [ ]:
[n for n,p in model.named_parameters()]

We want to make sure that we will only adjust the final ones, which we can figure out from the name:

In [ ]:
# we will set all of them to be not changed (requires_grad) except the final layer
# this tells pytorch not to calculate gradients for all the other weights
for name, param in model.named_parameters():
    param.requires_grad = name.startswith("fc")

Next we use a simple training loop. To keep a classroom run manageable, use 30 epochs with CUDA or 8 on CPU by default; feel free to experiment with these numbers. The model's pretrained layers remain frozen, so we only train the new final layer. The progress bar displays the validation accuracy, but do not interpret results from only two validation images per class as a robust estimate.


In [ ]:
from tqdm.auto import tqdm

# New weights in the final layer; the rest of ResNet is frozen.
model.fc.reset_parameters()
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.fc.parameters(), lr=1e-3)

# Change these if you have more time. CPU fallback remains usable.
num_epochs = 30 if device.type == "cuda" else 8
results = []

# Keep pretrained batch-normalisation statistics fixed during training.
model.eval()
for epoch in tqdm(range(1, num_epochs + 1)):
    for images, labels in train_loader:
        images = images.to(device, non_blocking=(device.type == "cuda"))
        labels = labels.to(device, non_blocking=(device.type == "cuda"))
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()

    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=(device.type == "cuda"))
            labels = labels.to(device, non_blocking=(device.type == "cuda"))
            outputs = model(images)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    results.append((epoch, val_acc))
    print(f"Epoch {epoch:02d} | validation accuracy {val_acc:.3f}")


With 12 classes, guessing uniformly at random would get roughly 1/12 correct on average. Compare your accuracy above with that baseline, remembering that the validation set has only 24 images. We have saved the accuracies in `results` so you can plot them later.

Let's pull three images randomly from the validation set and see what our trained model makes of them.


In [ ]:
model.eval()  # put the model in evaluation mode 

# boring matplotlib stuff
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Randomly choose 3 distinct indices from the validation subset
indices = random.sample(range(len(val_ds)), 3)

for ax, idx in zip(axes, indices):

    # get the image and the label as a number
    img, label = val_ds[idx]

    # the only thing we need to do with the label is to use it to look up the true label string
    true_label = dataset.classes[label]

    # turn off the gradient as we are not doing training
    with torch.no_grad():
        
        # Models expect a batch dimension: (N, C, H, W). `unsqueeze(0)` adds N=1.
        # We also move the input to the same device as the model.

        # run the model on the image; because it expects multiple images we have to package it with unsqueeze
        logits = model(img.unsqueeze(0).to(device))
        
        # easier for humans to look at probabilities so we softmax
        probs = torch.softmax(logits, dim=1)
        
        # the results are as a batch, so drop the extra dimension
        probs = probs.squeeze(0).cpu()  # move probabilities back for printing
        
        # Predicted class index is the argmax of probabilities (equivalently logits)
        predicted_class_number = int(probs.argmax())
        predicted_class = dataset.classes[predicted_class_number]

    
    # some boring manipulation to display the image
    # the order of the arrays is different between pytorch and matplotlib so reorder them
    npimg = img.permute(1, 2, 0).numpy()
    
    # and rescale the RBG values so they can be displayed
    npimg = (npimg - npimg.min()) / (npimg.max() - npimg.min() + 1e-8)

    # boring matplotlib code; show the image and predictions
    ax.imshow(npimg)
    ax.axis("off")
    ax.set_title(f"True: {true_label} Pred: {predicted_class}")

    print(f"Image idx {idx} — True: {true_label} | Pred: {predicted_class}")
    for class_idx, class_name in enumerate(dataset.classes):
        print(f"  {class_name:>20}: {probs[class_idx]:.3f}")

plt.tight_layout()
plt.show()


Try switching the device assignment above and rerunning the **image section** to compare CPU and GPU training times. Selecting a GPU runtime alone does not accelerate CPU tensors: the model and image batches must be on the same device. Rerun the whole notebook after changing the device, so the loaders and model use the same setting.
